# Séance 7 — Le package `medcalc` — correction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-medecine/notebooks/pym-s7-exercices-correction.ipynb)

Dans ce notebook, vous construisez un vrai petit package Python,
`medcalc`, qui regroupe des conversions d'unités de biologie et des
scores cliniques ; vous le testez avec **pytest** et vous le relisez
avec **pylint**.

**Objectifs**

- organiser du code en package (`__init__.py`, `pyproject.toml`,
  `pip install -e .`) ;
- écrire des tests unitaires : `assert`, `pytest.approx`,
  `pytest.raises`, `@pytest.mark.parametrize` ;
- trouver un bug grâce aux tests ;
- appliquer la PEP 8 et faire monter sa note `pylint`.

**Comment travailler**

- Exécutez les cellules **dans l'ordre**. Les cellules `%%writefile`
  écrivent un fichier sur le disque au lieu d'exécuter du code : après
  chaque modification d'un fichier, ré-exécutez sa cellule.
- Les cellules marquées `# À vous` sont à compléter.
- Chaque exercice se termine par une cellule de **vérification** qui
  affiche ✅ quand votre réponse est juste. Ne la modifiez pas.
- Pour repartir de zéro, ré-exécutez la cellule de préparation : elle
  efface et recrée le dossier de travail.

Les formules et seuils médicaux sont des exemples pédagogiques, pas des
outils d'aide à la décision clinique.

## Préparation

La cellule suivante crée un dossier de travail vide
(`/content/medcalc_projet` dans Colab), s'y place, et installe `pylint`
et `pytest-cov` s'ils manquent (`pytest` est déjà dans Colab).

In [ ]:
import importlib
import importlib.util
import os
import re
import shutil
import subprocess
import sys
import tempfile

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    WORKDIR = "/content/medcalc_projet"
else:
    WORKDIR = os.path.join(tempfile.gettempdir(), "medcalc_projet")

os.chdir(os.path.dirname(WORKDIR))
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(os.path.join(WORKDIR, "medcalc"))
os.makedirs(os.path.join(WORKDIR, "tests"))
os.makedirs(os.path.join(WORKDIR, "demo"))
os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

for paquet, module in [("pytest", "pytest"), ("pylint", "pylint"),
                       ("pytest-cov", "pytest_cov")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquet],
                       check=True)
print("Dossier de travail :", os.getcwd())

Quelques outils pour les cellules de vérification. Inutile de les lire
en détail : `lancer` exécute une commande Python (`pytest`, `pylint`)
dans le dossier du projet, `recharger` ré-importe `medcalc` après une
modification, et `bug_detecte` copie le projet, y injecte un bug, et
regarde si vos tests le détectent.

In [ ]:
def lancer(*arguments, dossier=None, afficher=True):
    """Exécute `python -m <arguments>` et renvoie le résultat."""
    resultat = subprocess.run([sys.executable, "-m", *arguments],
                              cwd=dossier or WORKDIR,
                              capture_output=True, text=True)
    if afficher:
        print(resultat.stdout + resultat.stderr)
    return resultat


def recharger(nom):
    """Ré-importe un module de medcalc après modification du fichier."""
    importlib.invalidate_caches()
    for module in list(sys.modules):
        if module == "medcalc" or module.startswith("medcalc."):
            del sys.modules[module]
    return importlib.import_module(nom)


def bug_detecte(fichier_test, module, code_bugge):
    """True si les tests de `fichier_test` échouent quand on ajoute
    `code_bugge` (une fonction redéfinie avec un bug) à `module`."""
    with tempfile.TemporaryDirectory() as copie:
        shutil.copytree(WORKDIR, copie, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns(
                            "__pycache__", ".pytest_cache", "*.egg-info"))
        with open(os.path.join(copie, module), "a", encoding="utf-8") as f:
            f.write("\n\n" + code_bugge + "\n")
        resultat = lancer("pytest", "-q", "-p", "no:cacheprovider",
                          fichier_test, dossier=copie, afficher=False)
    return resultat.returncode == 1


def note_pylint(cible):
    """Lance pylint sur `cible` et renvoie la note sur 10."""
    resultat = lancer("pylint", cible)
    note = re.search(r"rated at (-?[0-9.]+)/10", resultat.stdout)
    assert note, "pylint n'a pas rendu de note : lisez sa sortie."
    return float(note.group(1))

---

# Partie A — Du module au package

## Lire du code 1 — `__name__`

La cellule suivante écrit un fichier `demo/outils.py`. La cellule
d'après l'**importe**, puis le **lance** comme un script.

**Que va afficher ce code ? Répondez avant d'exécuter.** Combien de
lignes, et lesquelles, s'affichent lors de l'import ? lors du lancement ?

In [ ]:
%%writefile demo/outils.py
print("nom du module :", __name__)


def imc(poids_kg, taille_m):
    return poids_kg / taille_m ** 2


if __name__ == "__main__":
    print("IMC :", round(imc(70, 1.75), 1))

In [ ]:
sys.path.insert(0, os.path.join(WORKDIR, "demo"))
import outils
print("---- lancé comme un script ----")
!{sys.executable} demo/outils.py

*Réponse :* à l'import, une seule ligne, `nom du module : outils` — le
bloc `if __name__ == "__main__"` est ignoré. Lancé comme un script,
`__name__` vaut `"__main__"` : deux lignes, `nom du module : __main__`
puis `IMC : 22.9`.

## Le squelette du package

Voici les fichiers qui font de `medcalc` un package installable : le
`README.md`, le `pyproject.toml` et un premier `__init__.py`.

In [ ]:
%%writefile README.md
# medcalc

Calculs cliniques usuels : conversions d'unités de biologie (glycémie,
créatinine) et scores (IMC, clairance de Cockcroft-Gault, qSOFA).
Exemples pédagogiques : ce n'est pas un outil d'aide à la décision.

    pip install -e .

    >>> from medcalc import imc
    >>> round(imc(70, 1.75), 1)
    22.9

In [ ]:
%%writefile pyproject.toml
[project]
name = "medcalc"
version = "0.1.0"
description = "Calculs cliniques usuels (exemples pédagogiques)"
readme = "README.md"
requires-python = ">=3.10"
dependencies = []

[build-system]
requires = ["setuptools>=77.0.3"]
build-backend = "setuptools.build_meta"

[tool.setuptools.packages.find]
where = ["."]
include = ["medcalc*"]

In [ ]:
%%writefile medcalc/__init__.py
"""medcalc : calculs cliniques usuels (exemples pédagogiques)."""

__version__ = "0.1.0"

On installe le package en mode **éditable** : Python lira directement
les fichiers de `medcalc/`, toute modification est prise en compte
sans réinstaller. Dans Colab :

In [ ]:
if IN_COLAB:
    !{sys.executable} -m pip install -q -e .

medcalc = recharger("medcalc")
print("medcalc", medcalc.__version__, "importé depuis", medcalc.__file__)

Un noyau déjà démarré ne voit un package installé pendant la session
qu'après redémarrage ; c'est pourquoi la cellule de préparation a aussi
ajouté le dossier du projet à `sys.path`. Hors Colab, lancez
`pip install -e .` dans un terminal, depuis le dossier du projet.

---

## Exercice 1 — Le module `conversions`

Le laboratoire rend la glycémie en g/L, une publication internationale
en mmol/L ; la créatininémie en µmol/L, la formule de Cockcroft-Gault
l'attend en mg/dL. Complétez `medcalc/conversions.py` :

- `FACTEUR_GLUCOSE = 5.551` (1 g/L de glucose = 5,551 mmol/L, masse
  molaire 180,16 g/mol) et `FACTEUR_CREATININE = 88.4`
  (1 mg/dL = 88,4 µmol/L) ;
- les quatre fonctions de conversion ;
- chacune lève `ValueError` si la valeur reçue est négative (écrivez
  la vérification **une seule fois**, dans une petite fonction appelée
  par les quatre : pas de code dupliqué).

In [ ]:
%%writefile medcalc/conversions.py
"""Conversions d'unités de biologie médicale."""

FACTEUR_GLUCOSE = 5.551      # 1 g/L de glucose = 5.551 mmol/L
FACTEUR_CREATININE = 88.4    # 1 mg/dL de créatinine = 88.4 µmol/L


def _verifier_positive(valeur):
    """Lève ValueError si une concentration est négative."""
    if valeur < 0:
        raise ValueError(f"concentration négative : {valeur}")


def glycemie_g_vers_mmol(glycemie_g_l):
    """Convertit une glycémie de g/L en mmol/L."""
    _verifier_positive(glycemie_g_l)
    return glycemie_g_l * FACTEUR_GLUCOSE


def glycemie_mmol_vers_g(glycemie_mmol_l):
    """Convertit une glycémie de mmol/L en g/L."""
    _verifier_positive(glycemie_mmol_l)
    return glycemie_mmol_l / FACTEUR_GLUCOSE


def creatinine_umol_vers_mgdl(creatinine_umol):
    """Convertit une créatininémie de µmol/L en mg/dL."""
    _verifier_positive(creatinine_umol)
    return creatinine_umol / FACTEUR_CREATININE


def creatinine_mgdl_vers_umol(creatinine_mgdl):
    """Convertit une créatininémie de mg/dL en µmol/L."""
    _verifier_positive(creatinine_mgdl)
    return creatinine_mgdl * FACTEUR_CREATININE

In [ ]:
# Vérification
conv = recharger("medcalc.conversions")
assert abs(conv.glycemie_g_vers_mmol(1.0) - 5.551) < 1e-9
assert abs(conv.glycemie_mmol_vers_g(7.0) - 7.0 / 5.551) < 1e-9
assert abs(conv.creatinine_umol_vers_mgdl(88.4) - 1.0) < 1e-9
assert abs(conv.creatinine_mgdl_vers_umol(1.2) - 106.08) < 1e-9
for f in (conv.glycemie_g_vers_mmol, conv.glycemie_mmol_vers_g,
          conv.creatinine_umol_vers_mgdl, conv.creatinine_mgdl_vers_umol):
    try:
        f(-1)
    except ValueError:
        continue
    raise AssertionError(f"{f.__name__}(-1) devrait lever ValueError")
print("✅ Exercice 1 réussi")

## Exercice 2 — Le module `scores`

Complétez `medcalc/scores.py` :

- `imc(poids_kg, taille_m)` : poids / taille² ; `ValueError` si le
  poids ou la taille est nul ou négatif.
- `clairance_cockcroft_gault(age, poids_kg, creatinine_umol, femme)` :

$$\mathrm{Cl}_{cr} = \frac{(140 - \mathrm{age}) \times \mathrm{poids}}{72 \times \mathrm{creat}_{mg/dL}} \quad (\times 0{,}85 \ \mathrm{chez\ la\ femme})$$

  en mL/min. La créatinine arrive en µmol/L : réutilisez
  `creatinine_umol_vers_mgdl` du module `conversions` plutôt que de
  réécrire le facteur. `ValueError` si la créatinine est nulle ou
  négative.
- `qsofa(frequence_respiratoire, pression_systolique, glasgow)` : un
  point par critère du quick SOFA (Sepsis-3, Singer et al., JAMA 2016) :
  fréquence respiratoire ≥ 22/min, pression systolique ≤ 100 mmHg,
  altération de la conscience (score de Glasgow < 15).

In [ ]:
%%writefile medcalc/scores.py
"""Scores et indices cliniques (exemples pédagogiques)."""

from medcalc.conversions import creatinine_umol_vers_mgdl


def imc(poids_kg, taille_m):
    """Indice de masse corporelle en kg/m².

    Args:
        poids_kg: poids en kilogrammes, strictement positif.
        taille_m: taille en mètres, strictement positive.

    Returns:
        L'IMC, poids divisé par le carré de la taille.
    """
    if poids_kg <= 0 or taille_m <= 0:
        raise ValueError("le poids et la taille doivent être positifs")
    return poids_kg / taille_m ** 2


def clairance_cockcroft_gault(age, poids_kg, creatinine_umol, femme):
    """Clairance de la créatinine (Cockcroft-Gault), en mL/min.

    (140 - âge) x poids / (72 x créatinine en mg/dL), x 0.85 chez la femme.
    """
    if creatinine_umol <= 0:
        raise ValueError("la créatinine doit être positive")
    creatinine_mgdl = creatinine_umol_vers_mgdl(creatinine_umol)
    clairance = (140 - age) * poids_kg / (72 * creatinine_mgdl)
    if femme:
        clairance *= 0.85
    return clairance


def qsofa(frequence_respiratoire, pression_systolique, glasgow):
    """Score qSOFA (Sepsis-3), entre 0 et 3.

    Un point pour chaque critère : FR >= 22/min, PAS <= 100 mmHg,
    altération de la conscience (Glasgow < 15).
    """
    return (int(frequence_respiratoire >= 22)
            + int(pression_systolique <= 100)
            + int(glasgow < 15))

In [ ]:
# Vérification
scores = recharger("medcalc.scores")
assert abs(scores.imc(70, 1.75) - 22.857) < 1e-3
assert abs(scores.clairance_cockcroft_gault(60, 70, 88.4, False) - 77.78) < 0.01
assert abs(scores.clairance_cockcroft_gault(60, 70, 88.4, True) - 66.11) < 0.01
assert scores.qsofa(22, 100, 14) == 3
assert scores.qsofa(21, 101, 15) == 0
for args in [(70, 0), (0, 1.7)]:
    try:
        scores.imc(*args)
    except ValueError:
        continue
    raise AssertionError(f"imc{args} devrait lever ValueError")
print("✅ Exercice 2 réussi")

## Exercice 3 — Ce que le package expose

Pour l'instant, il faut écrire `from medcalc.scores import imc`.
Modifiez `medcalc/__init__.py` pour que `from medcalc import imc`
fonctionne aussi, pour `imc`, `clairance_cockcroft_gault` et `qsofa`.
Gardez `__version__`.

In [ ]:
%%writefile medcalc/__init__.py
"""medcalc : calculs cliniques usuels (exemples pédagogiques)."""

from medcalc.scores import clairance_cockcroft_gault, imc, qsofa

__version__ = "0.1.0"
__all__ = ["clairance_cockcroft_gault", "imc", "qsofa"]

In [ ]:
# Vérification
medcalc = recharger("medcalc")
from medcalc import clairance_cockcroft_gault, imc, qsofa
assert medcalc.__version__ == "0.1.0"
assert round(imc(80, 2.0), 1) == 20.0
print("✅ Exercice 3 réussi")

---

# Partie B — Tester avec pytest

## Lire du code 2 — quels tests pytest exécute-t-il ?

Le fichier ci-dessous contient trois fonctions. pytest ne collecte que
les fichiers `test_*.py` et, dedans, les fonctions dont le nom commence
par `test`.

**Que va afficher pytest ? Répondez avant d'exécuter.** Combien de
tests sont lancés, combien passent, combien échouent ?

In [ ]:
%%writefile demo/test_lecture.py
def test_addition():
    assert 2 + 2 == 4


def verifier_soustraction():
    assert 2 - 2 == 1


def test_flottants():
    assert 0.1 + 0.2 == 0.3

In [ ]:
!{sys.executable} -m pytest -q -p no:cacheprovider demo/test_lecture.py

*Réponse :* deux tests collectés (`verifier_soustraction` est ignorée,
son nom ne commence pas par `test`, alors qu'elle échouerait). `test_addition`
passe ; `test_flottants` **échoue** : `0.1 + 0.2` vaut
`0.30000000000000004` en virgule flottante. Lisez le rapport : la ligne
`>` montre l'`assert` en cause, la ligne `E` les valeurs comparées.
C'est pourquoi on compare des flottants avec `pytest.approx(0.3)`.

## Exercice 4 — Tester les conversions

Complétez `tests/test_conversions.py` :

- une valeur connue pour chaque fonction, comparée avec `pytest.approx` ;
- un aller-retour (g/L → mmol/L → g/L redonne la valeur de départ) ;
- un test qui vérifie, avec `pytest.raises(ValueError)`, que **chaque**
  conversion refuse une valeur négative (astuce : paramétrez le test
  par la fonction elle-même).

Lancez vos tests avec la cellule suivante. La vérification contrôle
ensuite qu'ils **détectent des bugs** : elle injecte tour à tour un
mauvais facteur de conversion, puis une conversion qui accepte les
valeurs négatives — vos tests doivent échouer dans les deux cas.

In [ ]:
%%writefile tests/test_conversions.py
import pytest

from medcalc.conversions import (
    creatinine_mgdl_vers_umol,
    creatinine_umol_vers_mgdl,
    glycemie_g_vers_mmol,
    glycemie_mmol_vers_g,
)


def test_glycemie_g_vers_mmol():
    assert glycemie_g_vers_mmol(1.0) == pytest.approx(5.551)
    assert glycemie_g_vers_mmol(1.26) == pytest.approx(7.0, abs=0.01)


def test_glycemie_mmol_vers_g():
    assert glycemie_mmol_vers_g(5.551) == pytest.approx(1.0)
    assert glycemie_mmol_vers_g(glycemie_g_vers_mmol(0.9)) == pytest.approx(0.9)


def test_creatinine_umol_vers_mgdl():
    assert creatinine_umol_vers_mgdl(88.4) == pytest.approx(1.0)
    assert creatinine_umol_vers_mgdl(0) == 0


def test_creatinine_mgdl_vers_umol():
    assert creatinine_mgdl_vers_umol(1.2) == pytest.approx(106.08)


@pytest.mark.parametrize("conversion", [
    glycemie_g_vers_mmol, glycemie_mmol_vers_g,
    creatinine_umol_vers_mgdl, creatinine_mgdl_vers_umol,
])
def test_valeur_negative_refusee(conversion):
    with pytest.raises(ValueError):
        conversion(-1)

In [ ]:
!{sys.executable} -m pytest -q tests/test_conversions.py

In [ ]:
# Vérification
resultat = lancer("pytest", "-q", "tests/test_conversions.py", afficher=False)
assert resultat.returncode == 0, "vos tests échouent sur le bon code : lisez le rapport"
assert bug_detecte(
    "tests/test_conversions.py", "medcalc/conversions.py",
    "def glycemie_g_vers_mmol(glycemie_g_l):\n    return glycemie_g_l * 5.0"), "un facteur faux (5.0) passe vos tests"
assert bug_detecte(
    "tests/test_conversions.py", "medcalc/conversions.py",
    "def creatinine_mgdl_vers_umol(creatinine_mgdl):\n"
    "    return creatinine_mgdl * 88.4"), "une créatinine négative est acceptée sans que vos tests le voient"
print("✅ Exercice 4 réussi")

## Exercice 5 — Tester les scores, avec `parametrize`

Complétez `tests/test_scores.py` :

- `test_imc` avec `pytest.approx` ; `test_imc_taille_invalide` avec
  `pytest.raises` ;
- `test_clairance_cockcroft_gault` pour un homme **et** une femme
  (homme de 60 ans, 70 kg, créatinine 88,4 µmol/L : 80 × 70 / 72 ≈
  77,8 mL/min) ;
- `test_qsofa` paramétré avec `@pytest.mark.parametrize`, au moins 5
  cas, dont les **valeurs seuils** FR = 22 et PAS = 100.

La vérification injecte trois bugs : un IMC qui divise par la taille au
lieu de son carré, une clairance qui oublie le facteur 0,85 chez la
femme, et un qSOFA écrit avec `> 22` au lieu de `>= 22`.

In [ ]:
%%writefile tests/test_scores.py
import pytest

from medcalc.scores import clairance_cockcroft_gault, imc, qsofa


def test_imc():
    assert imc(80, 2.0) == pytest.approx(20.0)
    assert imc(70, 1.75) == pytest.approx(22.857, abs=1e-3)


@pytest.mark.parametrize("poids, taille", [(70, 0), (70, -1.7), (0, 1.7)])
def test_imc_taille_invalide(poids, taille):
    with pytest.raises(ValueError):
        imc(poids, taille)


def test_clairance_cockcroft_gault():
    # homme de 60 ans, 70 kg, créatinine 88.4 µmol/L (1 mg/dL) : 80*70/72
    assert clairance_cockcroft_gault(60, 70, 88.4, femme=False) == \
        pytest.approx(77.78, abs=0.01)
    assert clairance_cockcroft_gault(60, 70, 88.4, femme=True) == \
        pytest.approx(77.78 * 0.85, abs=0.01)


def test_clairance_creatinine_invalide():
    with pytest.raises(ValueError):
        clairance_cockcroft_gault(60, 70, 0, femme=False)


@pytest.mark.parametrize("fr, pas, glasgow, attendu", [
    (16, 120, 15, 0),
    (21, 101, 15, 0),     # juste sous les seuils
    (22, 120, 15, 1),     # FR au seuil
    (16, 100, 15, 1),     # PAS au seuil
    (16, 120, 14, 1),
    (24, 95, 13, 3),
])
def test_qsofa(fr, pas, glasgow, attendu):
    assert qsofa(fr, pas, glasgow) == attendu

In [ ]:
!{sys.executable} -m pytest -v tests/test_scores.py

In [ ]:
# Vérification
resultat = lancer("pytest", "-q", "tests/test_scores.py", afficher=False)
assert resultat.returncode == 0, "vos tests échouent sur le bon code : lisez le rapport"
BUGS = {
    "IMC divisé par la taille": (
        "def imc(poids_kg, taille_m):\n"
        "    if poids_kg <= 0 or taille_m <= 0:\n"
        "        raise ValueError('poids ou taille invalide')\n"
        "    return poids_kg / taille_m"),
    "clairance sans 0,85 chez la femme": (
        "def clairance_cockcroft_gault(age, poids_kg, creatinine_umol, femme):\n"
        "    if creatinine_umol <= 0:\n"
        "        raise ValueError('creatinine invalide')\n"
        "    return (140 - age) * poids_kg * 88.4 / (72 * creatinine_umol)"),
    "qSOFA avec > 22": (
        "def qsofa(frequence_respiratoire, pression_systolique, glasgow):\n"
        "    return (int(frequence_respiratoire > 22)"
        " + int(pression_systolique <= 100) + int(glasgow < 15))"),
}
for nom, code_bugge in BUGS.items():
    assert bug_detecte("tests/test_scores.py", "medcalc/scores.py", code_bugge), \
        f"bug non détecté par vos tests : {nom}"
print("✅ Exercice 5 réussi")

## Exercice 6 — Trouvez le bug grâce aux tests

Un collègue a écrit `medcalc/tension.py`, qui classe une mesure de
pression artérielle selon les recommandations ESC/ESH 2018 :

| Catégorie | Systolique (mmHg) | | Diastolique (mmHg) |
|---|---|---|---|
| optimale | < 120 | et | < 80 |
| normale | 120–129 | et/ou | 80–84 |
| normale haute | 130–139 | et/ou | 85–89 |
| HTA grade 1 | 140–159 | et/ou | 90–99 |
| HTA grade 2 | 160–179 | et/ou | 100–109 |
| HTA grade 3 | ≥ 180 | et/ou | ≥ 110 |

Une mesure est classée dans la plus haute des deux catégories. Les
tests, eux, ont été écrits d'après ce tableau. Exécutez la cellule des
tests, puis lancez pytest : **lisez le rapport d'échec**, trouvez les
erreurs dans le code (il y en a deux) et corrigez la cellule
`%%writefile medcalc/tension.py`.

In [ ]:
%%writefile tests/test_tension.py
import pytest

from medcalc.tension import classer_tension


@pytest.mark.parametrize("systolique, diastolique, attendu", [
    (115, 75, "optimale"),
    (125, 78, "normale"),
    (118, 82, "normale"),
    (135, 80, "normale haute"),
    (145, 85, "HTA grade 1"),
    (130, 95, "HTA grade 1"),
    (165, 95, "HTA grade 2"),
    (180, 100, "HTA grade 3"),
    (150, 112, "HTA grade 3"),
])
def test_classer_tension(systolique, diastolique, attendu):
    assert classer_tension(systolique, diastolique) == attendu

In [ ]:
%%writefile medcalc/tension.py
"""Classification de la pression artérielle (ESC/ESH 2018)."""


def classer_tension(systolique, diastolique):
    """Catégorie ESC/ESH 2018 d'une mesure de pression artérielle en mmHg.

    La mesure est classée dans la plus haute des deux catégories
    (systolique ou diastolique).
    """
    if systolique >= 180 or diastolique >= 110:
        return "HTA grade 3"
    if systolique >= 160 or diastolique >= 100:
        return "HTA grade 2"
    if systolique >= 140 or diastolique >= 90:
        return "HTA grade 1"
    if systolique >= 130 or diastolique >= 85:
        return "normale haute"
    if systolique >= 120 or diastolique >= 80:
        return "normale"
    return "optimale"

In [ ]:
!{sys.executable} -m pytest -q tests/test_tension.py

In [ ]:
# Vérification
resultat = lancer("pytest", "-q", "tests/test_tension.py", afficher=False)
assert resultat.returncode == 0, "des tests échouent encore : lisez le rapport"
tension = recharger("medcalc.tension")
assert tension.classer_tension(180, 70) == "HTA grade 3"
assert tension.classer_tension(150, 70) == "HTA grade 1"
print("✅ Exercice 6 réussi")

*À noter :* le premier bug (`> 180`) ne se voit qu'avec un test **au
seuil** exact (180 mmHg) ; le second (`and` au lieu de `or`) avec une
mesure dont une seule des deux pressions est élevée.

---

# Partie C — PEP 8 et pylint

## Lire du code 3 — repérer les écarts à la PEP 8

Le fichier ci-dessous **fonctionne**, mais il est mal écrit.

**Avant d'exécuter pylint, listez au moins cinq écarts à la PEP 8.**
Puis exécutez les deux cellules et comparez votre liste aux messages
de pylint.

In [ ]:
%%writefile mauvais_code.py
import os, sys
import math
def CalculIMC(Poids,Taille) :
  l=Poids/(Taille*Taille)
  if l == None : return None
  return l
def Categorie( imc ):
  if imc<18.5: return 'insuffisance ponderale'
  elif imc<25: return "corpulence normale"
  elif imc<30 : return 'surpoids'
  else : return "obesite"

In [ ]:
!{sys.executable} -m pylint mauvais_code.py

*Réponse :* imports inutiles (`os`, `sys`, `math`) et plusieurs imports
sur une ligne ; noms en `CamelCase` pour des fonctions et des
arguments ; nom `l` ; indentation à 2 espaces ; `== None` au lieu de
`is None` ; espaces avant `:` et dans `Categorie( imc )` ; pas
d'espaces autour de `=` et `<` ; guillemets simples et doubles
mélangés ; aucune docstring ; pas de lignes vides entre les fonctions ;
`elif` après `return` inutiles. Chaque message de pylint donne
`fichier:ligne:colonne`, un code (C = convention, W = avertissement,
R = refactorisation, E = erreur) et une explication.

## Exercice 7 — Faire monter la note

Réécrivez ce code dans `imc_propre.py` :

- deux fonctions `calculer_imc(poids_kg, taille_m)` et
  `categorie_imc(imc)` (catégories de l'OMS : < 18,5 insuffisance
  pondérale, < 25 corpulence normale, < 30 surpoids, sinon obésité ;
  gardez exactement les chaînes renvoyées par le code d'origine) ;
- une docstring de module et une par fonction ;
- aucune ligne inutile, et un style PEP 8.

La vérification exige une note pylint d'au moins **9/10** et le même
comportement que le code d'origine.

In [ ]:
%%writefile imc_propre.py
"""Indice de masse corporelle et catégories de l'OMS."""


def calculer_imc(poids_kg, taille_m):
    """Renvoie l'IMC en kg/m² d'un poids en kg et d'une taille en m."""
    return poids_kg / taille_m ** 2


def categorie_imc(imc):
    """Renvoie la catégorie OMS correspondant à un IMC."""
    if imc < 18.5:
        return "insuffisance ponderale"
    if imc < 25:
        return "corpulence normale"
    if imc < 30:
        return "surpoids"
    return "obesite"

In [ ]:
# Vérification
note = note_pylint("imc_propre.py")
assert note >= 9, f"note pylint {note}/10 : lisez les messages ci-dessus"
sys.path.insert(0, WORKDIR)
import imc_propre
importlib.reload(imc_propre)
assert abs(imc_propre.calculer_imc(70, 1.75) - 22.857) < 1e-3
attendu = {17: "insuffisance ponderale", 18.5: "corpulence normale",
           24.9: "corpulence normale", 25: "surpoids", 30: "obesite"}
for valeur, categorie in attendu.items():
    assert imc_propre.categorie_imc(valeur) == categorie, valeur
print("✅ Exercice 7 réussi")

## Exercice 8 — Le package entier

Lancez pylint sur tout le package `medcalc`. Si la note est inférieure
à 9/10, corrigez les fichiers concernés (remontez aux cellules
`%%writefile` des exercices 1 à 6, modifiez-les et ré-exécutez-les),
puis relancez pylint **et les tests** : une correction de style ne
doit jamais changer un résultat.

In [ ]:
# Vérification
note = note_pylint("medcalc")
assert note >= 9, f"note pylint {note}/10 : lisez les messages ci-dessus"
resultat = lancer("pytest", "-q", "tests", afficher=False)
assert resultat.returncode == 0, "vos corrections ont cassé un test"
print("✅ Exercice 8 réussi")

`ruff` est une alternative récente à pylint, beaucoup plus rapide
(`pip install ruff` puis `ruff check .`) ; les deux suivent la PEP 8.

---

## Pour aller plus loin — Exercice 9 : la couverture des tests

Le plugin `pytest-cov` mesure la part des lignes de `medcalc` exécutées
par vos tests. Lancez la cellule suivante : la colonne `Miss` compte
les lignes jamais exécutées, `Missing` donne leurs numéros.

Ajoutez des tests (dans les cellules des exercices 4 et 5, ou dans un
nouveau fichier `tests/test_package.py`) jusqu'à atteindre **100 %**
de couverture. Pensez au `__init__.py` et aux branches qui lèvent
`ValueError`. Une couverture de 100 % ne prouve pas l'absence de bug :
elle garantit seulement qu'aucune ligne n'a jamais été exécutée par
aucun test.

In [ ]:
!{sys.executable} -m pytest -q --cov=medcalc --cov-report=term-missing tests

In [ ]:
# Vérification
resultat = lancer("pytest", "-q", "-p", "no:cacheprovider", "--cov=medcalc",
                  "--cov-report=term", "tests", afficher=False)
couverture = re.search(r"^TOTAL.*?(\d+)%$", resultat.stdout, re.MULTILINE)
assert resultat.returncode == 0, "des tests échouent"
assert couverture, resultat.stdout
assert int(couverture.group(1)) == 100, \
    f"couverture : {couverture.group(1)} %"
print("✅ Exercice 9 réussi")